# Flu Shot Learning: CatBoost + 5-Fold CV + Feature Engineering

This notebook implements a robust CatBoost model with 5-Fold Stratified Cross-Validation and advanced feature engineering to prevent overfitting and boost the ROC AUC score on the leaderboard.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data
Adjust the `DATA_DIR` path below if running directly on Kaggle.

In [2]:
DATA_DIR = '/kaggle/input/datasets/barshamishra19/flu-shot-learning-dataset'  # On Kaggle, change to '/kaggle/input/flu-shot-learning...'
SUBMISSIONS_DIR = '../submissions'
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

try:
    train_features = pd.read_csv(os.path.join(DATA_DIR, 'training_set_features.csv'), index_col='respondent_id')
    train_labels = pd.read_csv(os.path.join(DATA_DIR, 'training_set_labels.csv'), index_col='respondent_id')
    test_features = pd.read_csv(os.path.join(DATA_DIR, 'test_set_features.csv'), index_col='respondent_id')
except FileNotFoundError:
    # Fallback to Kaggle paths if running there
    DATA_DIR = '/kaggle/input/flu-shot-learning-h1n1-seasonal-flu-vaccines'
    train_features = pd.read_csv(os.path.join(DATA_DIR, 'training_set_features.csv'), index_col='respondent_id')
    train_labels = pd.read_csv(os.path.join(DATA_DIR, 'training_set_labels.csv'), index_col='respondent_id')
    test_features = pd.read_csv(os.path.join(DATA_DIR, 'test_set_features.csv'), index_col='respondent_id')


## 2. Feature Engineering
We create new aggregated features: the total number of missing values per row, the sum of behavioral preventative measures, and the sum of opinion/worry indicators. CatBoost will naturally handle any remaining missing values.

In [3]:
def feature_engineer(df):
    df = df.copy()
    
    # 1. Count Missing Values
    df['missing_count'] = df.isnull().sum(axis=1)
    
    # 2. Behavior Score: Sum of all behavioral binary flags
    behavior_cols = [col for col in df.columns if 'behavioral' in col]
    df['behavior_score'] = df[behavior_cols].sum(axis=1)
    
    # 3. Opinion Score: Sum of all opinion ratings
    opinion_cols = [col for col in df.columns if 'opinion' in col]
    df['opinion_score'] = df[opinion_cols].sum(axis=1)
    
    # CatBoost expects categorical features to be strings, so we cast object columns to string
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('Missing_Category').astype(str)
            
    return df

X = feature_engineer(train_features)
X_test = feature_engineer(test_features)
y = train_labels

# Identify categorical features for CatBoost
cat_features = [col for col in X.columns if X[col].dtype == 'object']
print(f"Categorical features: {len(cat_features)}")

Categorical features: 12


## 3. 5-Fold Stratified Cross-Validation & Modeling
We train 5 separate CatBoost models per target variable. This prevents overfitting and gives us a robust out-of-fold (OOF) score that reliably reflects the Kaggle leaderboard.

In [4]:
def train_and_predict(target_name):
    print(f"\n{'='*40}")
    print(f"Training models for {target_name}")
    print(f"{'='*40}")
    
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    y_target = y[target_name]
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y_target)):
        X_tr, y_tr = X.iloc[train_idx], y_target.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y_target.iloc[val_idx]
        
        model = CatBoostClassifier(
            iterations=1500, 
            learning_rate=0.03, 
            depth=6, 
            eval_metric='AUC', 
            random_seed=42,
            cat_features=cat_features,
            early_stopping_rounds=150,
            verbose=200
        )
        
        model.fit(
            X_tr, y_tr, 
            eval_set=(X_va, y_va), 
            use_best_model=True
        )
        
        # Predict on validation fold
        val_preds = model.predict_proba(X_va)[:, 1]
        oof_preds[val_idx] = val_preds
        
        # Predict on test set and average across folds
        test_preds += model.predict_proba(X_test)[:, 1] / kf.n_splits
        
        fold_auc = roc_auc_score(y_va, val_preds)
        print(f"--> Fold {fold+1} AUC: {fold_auc:.4f}\n")
        
    final_auc = roc_auc_score(y_target, oof_preds)
    print(f"*** Final OOF AUC for {target_name}: {final_auc:.4f} ***\n")
    
    return test_preds, final_auc

h1n1_preds, h1n1_auc = train_and_predict('h1n1_vaccine')
seasonal_preds, seasonal_auc = train_and_predict('seasonal_vaccine')

print(f"\n{'='*40}")
print(f"OVERALL MEAN ROC AUC: {(h1n1_auc + seasonal_auc) / 2:.4f}")
print(f"{'='*40}")


Training models for h1n1_vaccine
0:	test: 0.8216513	best: 0.8216513 (0)	total: 94.5ms	remaining: 2m 21s
200:	test: 0.8664196	best: 0.8664230 (199)	total: 6.86s	remaining: 44.4s
400:	test: 0.8696892	best: 0.8697635 (393)	total: 13.3s	remaining: 36.6s
600:	test: 0.8711401	best: 0.8711736 (594)	total: 20.1s	remaining: 30.1s
800:	test: 0.8715407	best: 0.8715558 (788)	total: 27.1s	remaining: 23.6s
1000:	test: 0.8716792	best: 0.8716953 (935)	total: 34s	remaining: 17s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.8716952761
bestIteration = 935

Shrink model to first 936 iterations.
--> Fold 1 AUC: 0.8717

0:	test: 0.8281116	best: 0.8281116 (0)	total: 36.6ms	remaining: 54.9s
200:	test: 0.8752293	best: 0.8752400 (198)	total: 6.76s	remaining: 43.7s
400:	test: 0.8782826	best: 0.8782826 (400)	total: 13.5s	remaining: 37.1s
600:	test: 0.8785278	best: 0.8787117 (508)	total: 20.4s	remaining: 30.5s
800:	test: 0.8792790	best: 0.8792790 (800)	total: 27.3s	remaining: 23.8s
1000:	te

## 4. Generate Submission
We populate the sample submission format with our robust cross-validated predictions and save it to the output folder.

In [5]:
# Load submission format
try:
    submission_df = pd.read_csv(os.path.join(DATA_DIR, 'submission_format.csv'), index_col='respondent_id')
except FileNotFoundError:
    submission_df = pd.read_csv(os.path.join(DATA_DIR, 'submission_format.csv'), index_col='respondent_id')

# Populate predictions
submission_df['h1n1_vaccine'] = h1n1_preds
submission_df['seasonal_vaccine'] = seasonal_preds

# Save submission
submission_path = os.path.join(SUBMISSIONS_DIR, '2flu_shot_submission.csv')
submission_df.to_csv(submission_path)
print(f"Submission successfully saved to {submission_path}")

Submission successfully saved to ../submissions/2flu_shot_submission.csv
